# Figure 6d. Genome-wide map of AD colocalization signals

Figure 6d is a Manhattan plot of the AD GWAS (Bellenguez et al.) with tracks marking AD colocalization confidence sets (CoS) shared with each QTL class: *cis*-eQTL, *trans*-eGene-QTL, *trans*-eProgram-QTL, protein-QTL, glycoprotein-QTL and metabolite-QTL. The Manhattan layer needs genome-wide GWAS summary statistics, which are available from the original study and are not redistributed here. This notebook does not draw the panel; it introduces the TransMap AD colocalization table behind the colocalization tracks.

Input: [`Supplementary_Data/AD_TransMap_colocalization.tsv`](../../Supplementary_Data/index.md). Column descriptions are in the Supplementary Data page.

## Load the table

Each row is one AD-focal 95% CoS from ColocBoost, retained by `get_robust_colocalization(cos_npc_cutoff = 0.5, npc_outcome_cutoff = 0.2, pvalue_cutoff = 1e-4)`. Multi-value fields are separated by `; `.

In [1]:
ad <- read.delim("../../Supplementary_Data/AD_TransMap_colocalization.tsv", stringsAsFactors = FALSE)
dim(ad)
str(ad, vec.len = 1, nchar.max = 80)

[1] 237  14

'data.frame':	237 obs. of  14 variables:
 $ focal_outcome            : chr  "AD_Bellenguez_2026" ...
 $ colocalized_outcomes     : chr  "adjusted_glyco_trans_biomarker_group4_adjusted_glyco_trans_gpQT"| __truncated__ ...
 $ cos_id                   : chr  "cos1:y5_y6_y7_y8_y9_y10_y11_y12_y14_y15_y16_y17" ...
 $ purity                   : num  0.943 ...
 $ top_variable             : chr  "chr1:114365210:A:T" ...
 $ top_variable_vcp         : num  0.32 ...
 $ cos_npc                  : num  0.997 ...
 $ min_npc_outcome          : num  0.248 ...
 $ n_variables              : int  5 5 ...
 $ colocalized_index        : chr  "5735; 5727; 5717; 5715; 5738" ...
 $ colocalized_variables    : chr  "chr1:114365210:A:T; chr1:114360677:G:A; chr1:114355480:T:C; chr"| __truncated__ ...
 $ colocalized_variables_vcp: chr  "0.319519722564465; 0.267120161649097; 0.252766835572435; 0.0933"| __truncated__ ...
 $ region                   : chr  "chr1:111177234-115035572" ...
 $ minpv                    : ch

In [2]:
## a compact view of one CoS per row
show <- c("top_variable", "top_variable_vcp", "purity", "cos_npc", "n_variables", "region")
head(ad[, show])

,top_variable,top_variable_vcp,purity,cos_npc,n_variables,region
,<chr>,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,chr1:114365210:A:T,0.3195197,0.9434331,0.9972,5,chr1:111177234-115035572
2,chr1:114365210:A:T,0.3033275,0.9426491,0.9980,5,chr1:112696831-115995370
3,chr1:114365210:A:T,0.2854842,0.9426491,0.9974,5,chr1:113837226-117135240
4,chr1:161189357:C:T,0.9427340,0.9901564,1.0000,2,chr1:155637783-162678654
5,chr1:161189357:C:T,0.9317810,0.9882142,1.0000,2,chr1:160399408-163599842
6,chr1:201012797:T:C,0.1415493,0.7234968,1.0000,39,chr1:197638456-202429866


## Rows, regions and signals

ColocBoost was run in overlapping analysis regions, so one signal can be reported in more than one region. The top variant is a convenient key for distinct signals.

In [3]:
c(CoS_rows = nrow(ad),
  analysis_regions = length(unique(ad$region)),
  distinct_top_variants = length(unique(ad$top_variable)))
summary(ad[, c("purity", "cos_npc", "n_variables", "top_variable_vcp")])

CoS_rows      analysis_regions distinct_top_variants 
                  237                   194                   133

     purity          cos_npc        n_variables     top_variable_vcp  
 Min.   :0.7080   Min.   :0.5869   Min.   :   1.0   Min.   :0.001161  
 1st Qu.:0.8783   1st Qu.:0.9972   1st Qu.:   3.0   1st Qu.:0.075372  
 Median :0.9513   Median :1.0000   Median :  14.0   Median :0.154949  
 Mean   :0.9270   Mean   :0.9845   Mean   :  41.4   Mean   :0.333099  
 3rd Qu.:0.9850   3rd Qu.:1.0000   3rd Qu.:  26.0   3rd Qu.:0.520441  
 Max.   :1.0000   Max.   :1.0000   Max.   :1948.0   Max.   :1.000000  

## One row per CoS and colocalized trait

Split `colocalized_outcomes` and `minpv` into long format and assign each trait to the QTL classes shown as tracks in Figure 6d.

In [4]:
outcomes <- strsplit(ad$colocalized_outcomes, "; ", fixed = TRUE)
minpv    <- strsplit(ad$minpv, "; ", fixed = TRUE)
long <- data.frame(
  cos_row = rep(seq_len(nrow(ad)), lengths(outcomes)),
  outcome = unlist(outcomes),
  minpv   = as.numeric(unlist(minpv)),
  stringsAsFactors = FALSE
)
qtl_class <- function(x) {
  ifelse(grepl("^AD_", x), "AD GWAS",
  ifelse(grepl("_cis_gene_", x), "cis-eQTL",
  ifelse(grepl("_trans_gene_", x), "trans-eGene-QTL",
  ifelse(grepl("_trans_local_program_", x), "trans-eProgram-QTL",
  ifelse(grepl("^protein_", x), "protein-QTL",
  ifelse(grepl("glyco", x), "glycoprotein-QTL",
  ifelse(grepl("^metabolomics_", x), "metabolite-QTL", "other")))))))
}
long$class <- qtl_class(long$outcome)
## context (cell type or bulk region) for transcriptomic traits
long$context <- ifelse(long$class %in% c("cis-eQTL", "trans-eGene-QTL", "trans-eProgram-QTL"),
                       sub("_.*", "", long$outcome), NA)
head(long)
table(long$class)

,cos_row,outcome,minpv,class,context
,<int>,<chr>,<dbl>,<chr>,<chr>
1,1,adjusted_glyco_trans_biomarker_group4_adjusted_glyco_trans_gpQTL_adj_gp_2659,2.756816e-05,glycoprotein-QTL,NA
2,1,adjusted_glyco_trans_biomarker_group4_adjusted_glyco_trans_gpQTL_adj_gp_3614,3.773104e-05,glycoprotein-QTL,NA
3,1,protein_trans_biomarker_group5_protein_trans_pQTL_ENSG00000166342,6.100533e-05,protein-QTL,NA
4,1,AC_trans_gene_group6_AC_ENSG00000100604,3.922235e-06,trans-eGene-QTL,AC
5,1,AC_trans_gene_group6_AC_ENSG00000106113,4.862208e-05,trans-eGene-QTL,AC
6,1,AC_trans_gene_group6_AC_ENSG00000112818,6.203700e-05,trans-eGene-QTL,AC



           AD GWAS           cis-eQTL   glycoprotein-QTL     metabolite-QTL 
               237                725                178                 10 
       protein-QTL    trans-eGene-QTL trans-eProgram-QTL 
               442              11468               1553 

## How many CoS involve each QTL class

These counts correspond to the colocalization tracks in Figure 6d (here counted per CoS row and per distinct top variant).

In [5]:
classes <- c("cis-eQTL", "trans-eGene-QTL", "trans-eProgram-QTL",
             "protein-QTL", "glycoprotein-QTL", "metabolite-QTL")
has <- sapply(classes, function(k) tapply(long$class == k, long$cos_row, any))
top <- ad$top_variable
data.frame(
  CoS_rows = colSums(has),
  distinct_top_variants = sapply(classes, function(k) length(unique(top[has[, k]])))
)

,CoS_rows,distinct_top_variants
,<dbl>,<int>
cis-eQTL,171,98
trans-eGene-QTL,229,128
trans-eProgram-QTL,208,119
protein-QTL,110,62
glycoprotein-QTL,100,58
metabolite-QTL,7,4


In [6]:
## transcriptomic traits by context
with(long[!is.na(long$context), ], table(context, class))

       class
context cis-eQTL trans-eGene-QTL trans-eProgram-QTL
  AC         153            2575                238
  Ast         59             324                126
  DLPFC      115            2453                285
  Exc         87            1717                168
  Inh         54            1107                192
  Mic         62             240                 85
  OPC         20             299                105
  Oli         53             332                111
  PCC        122            2421                243

## CoS per chromosome

The chromosome positions of the colocalization tracks come from the CoS variants (`colocalized_variables`, GRCh38).

In [7]:
ad$chr <- factor(sub(":.*", "", ad$top_variable), levels = paste0("chr", 1:22))
ad$pos  <- as.numeric(sapply(strsplit(ad$top_variable, ":"), `[`, 2))
table(ad$chr)


 chr1  chr2  chr3  chr4  chr5  chr6  chr7  chr8  chr9 chr10 chr11 chr12 chr13 
   12    22     5     6    13     6    17    20     5    17    11    11     3 
chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 chr22 
   11    12    17    19     0    14     6     8     2 

## Marginal p-values

`minpv` is the smallest marginal p-value over the reported CoS variants for each trait. Count the trait entries above 1e-4.

In [8]:
ok <- long$class != "AD GWAS"
c(trait_entries = sum(ok), above_1e4 = sum(long$minpv[ok] > 1e-4), max = max(long$minpv[ok]))

trait_entries     above_1e4           max 
  1.43760e+04   7.20000e+01   9.69566e-03

## Example: look up a locus

The chr16 INO80E region from Figure 7e.

In [9]:
ino <- grepl("ENSG00000169592", ad$colocalized_outcomes)   # INO80E
ad[ino, c("top_variable", "n_variables", "purity", "cos_npc", "region")]
sort(table(long$class[long$cos_row %in% which(ino)]), decreasing = TRUE)

,top_variable,n_variables,purity,cos_npc,region
,<chr>,<int>,<dbl>,<dbl>,<chr>
83,chr16:29951481:G:A,32,0.9825863,1,chr16:24031743-30613717
85,chr16:29951481:G:A,32,0.9825863,1,chr16:27341433-34991551



   trans-eGene-QTL           cis-eQTL trans-eProgram-QTL   glycoprotein-QTL 
                46                 31                 10                  5 
           AD GWAS 
                 2 